# Préparation des données MediQAl

## Objectif du notebook

Ce notebook a pour objectif de préparer les données médicales francophones de MediQAl afin de construire un benchmark homogène, contrôlé et directement exploitable pour l’évaluation des grands modèles de langage.

À partir des fichiers Parquet produits lors de l’exploration, le notebook réalise les étapes suivantes :

1. charger séparément les splits `train`, `validation` et `test` ;
2. vérifier la structure, les colonnes et l’intégrité des données ;
3. effectuer un nettoyage léger sans modifier le sens médical des questions ;
4. formater les propositions des questions à choix multiple ;
5. construire la réponse de référence à partir de la lettre correcte ;
6. préparer le contexte textuel qui sera utilisé dans les futurs prompts ;
7. valider la qualité, l’unicité et la cohérence des données ;
8. enregistrer les jeux de données préparés au format Parquet.

Le traitement conserve les splits officiels afin d’éviter toute fuite de données :

- le split `train` servira au développement ou à l’entraînement du détecteur ;
- le split `validation` servira au choix des prompts, des paramètres et des seuils ;
- le split `test` sera réservé à l’évaluation finale.

Dans une première version du projet, la préparation porte principalement sur la configuration `MCQU`, qui correspond aux questions à choix unique. Ce choix permet de disposer d’une vérité terrain claire et de mettre en place une première évaluation reproductible des réponses générées par les LLM.

## Pipeline de préparation

```mermaid
flowchart TD
    A["Fichiers Parquet bruts MCQU"] --> B["Chargement des splits"]
    B --> C["Contrôles d'intégrité"]
    C --> D["Nettoyage léger"]
    D --> E["Formatage des choix"]
    E --> F["Construction du contexte"]
    F --> G["Validation des données"]
    G --> H["Fichiers Parquet préparés"]

## 1. Importations et chemins

In [17]:
from pathlib import Path

import pandas as pd
import numpy as np

In [18]:
# définir le chemin d'importation et d'enregistrement des données
PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "medical"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## 2. Chargement des splits

In [19]:
splits = ["train", "validation", "test"]

dfs_rw = {
    split: pd.read_parquet(
        RAW_DIR / f"medical_mcqu_{split}.parquet"
    )
    for split in splits
}

In [20]:
# vérification des dimensions des DataFrames
for split, df in dfs_rw.items():
    print(f"{split.capitalize()} set shape: {df.shape}")

Train set shape: (10113, 14)
Validation set shape: (2561, 14)
Test set shape: (4343, 14)


In [21]:
# vérification du schéma des DataFrames
for split, df in dfs_rw.items():
    print(f"\n{split.capitalize()} set schema:")
    print(df.dtypes)


Train set schema:
id                       object
clinical_case            object
question                 object
answer_a                 object
answer_b                 object
answer_c                 object
answer_d                 object
answer_e                 object
correct_answers          object
task                     object
medical_subject          object
question_type            object
question_length_chars     int64
question_length_words     int64
dtype: object

Validation set schema:
id                       object
clinical_case            object
question                 object
answer_a                 object
answer_b                 object
answer_c                 object
answer_d                 object
answer_e                 object
correct_answers          object
task                     object
medical_subject          object
question_type            object
question_length_chars     int64
question_length_words     int64
dtype: object

Test set schema:
id             

## 4. Ne pas supprimer les cas cliniques manquants

Une valeur manquante dans *clinical_case* ne signifie pas forcément que la ligne est inutilisable. Certaines questions médicales ne nécessitent simplement pas de cas clinique.

Remplacez ces valeurs par une chaîne vide :

In [22]:


def clean_text_column(series):
    return (
        series
        .fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

## 5. Préparation des choix de réponses

On veut associer les lettres "A, B, ..." avec les propositions de réponse "answer_a, answer_b, ..."

In [23]:
CHOICE_COLUMNS = {
    "A": "answer_a",
    "B": "answer_b",
    "C": "answer_c",
    "D": "answer_d",
    "E": "answer_e",
}

def build_choices(row):
    return {
        letter: row[column]
        for letter, column in CHOICE_COLUMNS.items()
    }

# dont une destinée au prompt 
def format_choices(row):
    return "\n".join(
        f"{letter}. {row[column]}"
        for letter, column in CHOICE_COLUMNS.items()
    )

## 6. Construction de la réponse de référence

MediQAl fournit la lettre correcte dans correct_answers. Il faut également récupérer le texte associé à la réponse dans la colonne cible.

In [24]:
def get_reference_answer(row):
    letter = row["correct_answers"].strip().upper()
    column = CHOICE_COLUMNS.get(letter)

    if column is None:
        return None

    return row[column]


## 7. Construction de la question avec les choix de réponses

In [25]:

def build_question_context(row):
    parts = []

    if row["clinical_case"]:
        parts.append(f"Cas clinique :\n{row['clinical_case']}")

    parts.append(f"Question :\n{row['question']}")
    parts.append(f"Propositions :\n{row['choices_text']}")

    return "\n\n".join(parts)

Exemple :

Cas clinique :
Un homme de 63 ans est hospitalisé...

Question :
Pour confirmer le diagnostic de maladie d’Addison...

Propositions :
A. Test de stimulation...
B. Test de freinage...
C. ...

##8. Création 

Création du dataset préparé

In [26]:
def prepare_mcqu(df, split):
    prepared = df.copy()

    text_columns = [
        "clinical_case",
        "question",
        "answer_a",
        "answer_b",
        "answer_c",
        "answer_d",
        "answer_e",
        "medical_subject",
        "question_type",
        "task",
    ]

    for column in text_columns:
        prepared[column] = clean_text_column(
            prepared[column]
        )

    prepared["reference_letter"] = (
        prepared["correct_answers"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    prepared["choices"] = prepared.apply(
        build_choices,
        axis=1
    )

    prepared["choices_text"] = prepared.apply(
        format_choices,
        axis=1
    )

    prepared["reference_answer"] = prepared.apply(
        get_reference_answer,
        axis=1
    )

    prepared["question_context"] = prepared.apply(
        build_question_context,
        axis=1
    )

    prepared["split"] = split
    prepared["configuration"] = "mcqu"

    prepared["sample_id"] = (
        "mcqu_"
        + split
        + "_"
        + prepared["id"].astype(str)
    )

    return prepared

In [27]:
# application au jeu de données complet
dfs_processed = {
    split: prepare_mcqu(df, split)
    for split, df in dfs_rw.items()
}

## 9. Selection des colonnes finales
Le but est de sélectionner les colonnes finales pour l'exportation. On peut choisir de conserver uniquement les colonnes pertinantes pour l'évaluation des modèles LLM.


In [28]:
final_columns = [
    "sample_id",
    "id",
    "configuration",
    "split",
    "clinical_case",
    "question",
    "answer_a",
    "answer_b",
    "answer_c",
    "answer_d",
    "answer_e",
    "choices",
    "choices_text",
    "reference_letter",
    "reference_answer",
    "medical_subject",
    "question_type",
    "task",
    "question_context",
]

# avec :
# - "configuration" : le type de question (ici "mcqu" pour "multiple choice question")
# - "choices" : un dictionnaire des propositions de réponses
# - "choices_text" : une chaîne de caractère formattée des propositions de réponses.
# - "question_context" : une chaîne de caractère formattée du contexte de la question, incluant le cas clinique, la question et les propositions" 


In [29]:
for split in dfs_processed:
    dfs_processed[split] = (
        dfs_processed[split][final_columns]
    )

## 10. Contrôles de qualité

In [30]:
# unicité des identifiants 
for split, df in dfs_processed.items():
    if df["sample_id"].duplicated().any():
        raise ValueError(f"Duplicate sample_id found in {split} set.")

In [31]:
# validité des réponses
valid_letters = {"A", "B", "C", "D", "E"}

for split, df in dfs_processed.items():
    invalid = ~df["reference_letter"].isin(valid_letters)

    print(
        split,
        "réponses invalides :",
        invalid.sum()
    )

train réponses invalides : 0
validation réponses invalides : 0
test réponses invalides : 0


In [32]:
# Réponse de référence manquante
for split, df in dfs_processed.items():
    print(
        split,
        "références manquantes :",
        df["reference_answer"].isna().sum()
    )

train références manquantes : 0
validation références manquantes : 0
test références manquantes : 0
